In [5]:
!pip install chromadb

In [6]:
# Your "Database" of documents
faq_database = [
    "To reset your password, click the 'Forgot Password' link on the login screen.",
    "Our refund policy allows returns within 30 days of the original purchase date.",
    "You can update your billing address in the 'Account Settings' dashboard."
]

# A messy user search query
user_query = "How do I get my money back for this thing?"


In [ ]:
import chromadb

# 1. Initialize the Database Client
# (This creates an in-memory database that clears when you close Python. 
# In production, you would use chromadb.PersistentClient to save it to disk).
client = chromadb.Client()

# 2. Create a "Collection" (This is the Vector DB equivalent of an SQL Table)
# We will call it "customer_support"
collection = client.create_collection(name="customer_support")

# 3. Add data to the database
# ChromaDB expects three distinct lists of equal length:
# - documents: The actual text strings
# - metadatas: Dictionaries for filtering (e.g., {"topic": "billing"})
# - ids: Unique string IDs for each row (e.g., "id1", "id2")
collection.add(
    documents=[
         "To reset your password, click the 'Forgot Password' link on the login screen.",
            "Our refund policy allows returns within 30 days of the original purchase date.",
            "You can update your billing address in the 'Account Settings' dashboard."
    ],
    metadatas=[
        {"source": "password_reset_page"}, 
        {"source": "refund_policy_page"},
        {"source": "account_settings_page"}
    ],
    ids=[
        "faq_1", 
        "faq_2",
        "faq_3"
    ]
)

# 4. Execute the Semantic Search
# Notice how we don't calculate Cosine Similarity manually anymore!
# The HNSW engine does it for us instantly. We just ask for the top 1 result (n_results=1)


InternalError: Collection [customer_support] already exists

In [14]:
results = collection.query(
    query_texts=["How do I reset my password and then get a refund?"],
    n_results=2
)

# 5. Print the winning document
print(results['documents'])

[["To reset your password, click the 'Forgot Password' link on the login screen.", 'Our refund policy allows returns within 30 days of the original purchase date.']]


In [15]:
collection = client.create_collection(name="regional_policies")

# 2. Add our conflicting documents with specific metadata
collection.add(
    documents=[
        "Our US refund policy allows returns within 30 days.",
        "Our European refund policy allows returns within 14 days by law."
    ],
    metadatas=[
        {"region": "US"}, 
        {"region": "EU"}
    ],
    ids=["doc_us", "doc_eu"]
)

# --- QUERY 1: Pure Vector Search ---
print("--- Result WITHOUT Filter ---")
unfiltered_results = collection.query(
    query_texts=["What is the refund policy?"],
    n_results=2
)
# This will return BOTH documents because mathematically, they both answer the question perfectly.
for doc in unfiltered_results['documents'][0]:
    print(f"- {doc}")

# --- QUERY 2: Vector Search + Metadata Filter ---
print("\n--- Result WITH 'where' Filter ---")
filtered_results = collection.query(
    query_texts=["What is the refund policy?"],
    n_results=2,
    where={"region": "EU"}  # <--- This is the exact syntax!
)
# This forces the HNSW graph to completely ignore the US document, even though it matches the keyword "refund".
for doc in filtered_results['documents'][0]:
    print(f"- {doc}")

--- Result WITHOUT Filter ---
- Our US refund policy allows returns within 30 days.
- Our European refund policy allows returns within 14 days by law.

--- Result WITH 'where' Filter ---
- Our European refund policy allows returns within 14 days by law.
